In [120]:
import pandas as pd
from sqlalchemy import create_engine

# Below we specify the parameters required to access the db
host = "localhost"
user = "root"
password = ""
database = "milk_collection_db"

# create a connection to the dd
connection = f"mysql+pymysql://{user}:{password}@{host}/{database}"

# By use of sqlalchemy engine establish the connection
engine = create_engine(connection)

import pandas as pd
from sqlalchemy import create_engine

# Below we specify the parameters required to access the db
host = "localhost"
user = "root"
password = ""
database = "milk_collection_db"

# create a connection to the dd
connection = f"mysql+pymysql://{user}:{password}@{host}/{database}"

# By use of sqlalchemy engine establish the connection
engine = create_engine(connection)

# specify the query you want to execute
query = """SELECT f.first_name AS farmer_first_name, f.last_name as farmer_last_name, p.amount, p.payment_date, p.status,c.litres * c.price_per_litre as Exact_amout_paid, c.litres, c.price_per_litre, po.first_name AS porter_first_name, po.last_name as porter_last_name FROM  farmers f 
INNER JOIN payment p ON f.farmer_id = p.farmer_id 
INNER JOIN collections c ON f.farmer_id = c.farmer_id
INNER JOIN porters po ON c.porter_id = po.porter_id;"""

# create a dataframe after the query has executed.
df = pd.read_sql_query(query, engine)

# print out the dataframe
df




,farmer_first_name,farmer_last_name,amount,payment_date,status,Exact_amout_paid,litres,price_per_litre,porter_first_name,porter_last_name
0,David,Maina,5730.0,2026-06-05,PAID,2730.0,45.5,60.0,Simon,Ouma
1,David,Maina,5730.0,2026-06-05,PAID,3000.0,50.0,60.0,Simon,Ouma
2,Stella,Nyaboke,4800.0,2026-06-05,PAID,2280.0,38.0,60.0,Simon,Ouma
3,Stella,Nyaboke,4800.0,2026-06-05,PAID,2520.0,42.0,60.0,Simon,Ouma
4,Richard,Anyango,6600.0,2026-06-05,PAID,3120.0,52.0,60.0,Ezra,Nehemiah
5,Richard,Anyango,6600.0,2026-06-05,PAID,3480.0,58.0,60.0,Ezra,Nehemiah
6,Lilian,Nasimiyu,4830.0,2026-06-05,PAID,2490.0,41.5,60.0,Ezra,Nehemiah
7,Lilian,Nasimiyu,4830.0,2026-06-05,PAID,2340.0,39.0,60.0,Ezra,Nehemiah
8,Duncan,Kiplagat,7380.0,2026-06-05,PAID,3600.0,60.0,60.0,Job,Mwangi
9,Duncan,Kiplagat,7380.0,2026-06-05,PAID,3780.0,63.0,60.0,Job,Mwangi


In [121]:
# finding the shape of our dafaframe
df.shape



(21, 10)

In [122]:
# check for some info in regards to the generated dataframe
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   farmer_first_name  21 non-null     str    
 1   farmer_last_name   21 non-null     str    
 2   amount             21 non-null     float64
 3   payment_date       21 non-null     object 
 4   status             21 non-null     str    
 5   Exact_amout_paid   21 non-null     float64
 6   litres             21 non-null     float64
 7   price_per_litre    21 non-null     float64
 8   porter_first_name  21 non-null     str    
 9   porter_last_name   21 non-null     str    
dtypes: float64(4), object(1), str(5)
memory usage: 1.8+ KB


In [123]:
# summary statistics for numeric columns
df.describe()

,amount,Exact_amout_paid,litres,price_per_litre
count,21.000000,21.000000,21.000000,21.000000
mean,6011.428571,2946.666667,50.500000,58.809524
std,1108.518316,553.446776,10.072239,5.455447
min,4350.000000,2100.000000,35.000000,35.000000
25%,4830.000000,2490.000000,42.000000,60.000000
50%,5970.000000,2910.000000,50.000000,60.000000
75%,6720.000000,3420.000000,58.000000,60.000000
max,7620.000000,3900.000000,70.000000,60.000000


In [124]:
# Below is how we export the entire dataframe into a csv
df.to_csv("milk_collection_data.csv", index=False)

In [126]:
# create a database table from an existing dataframe
df.to_sql("collection_summary", engine, if_exists="replace", index=False)

21

In [127]:
# check missing and duplicate values.
print(df.isnull().sum())
print("Duplicate rows: ", df.duplicated().sum())

farmer_first_name    0
farmer_last_name     0
amount               0
payment_date         0
status               0
Exact_amout_paid     0
litres               0
price_per_litre      0
porter_first_name    0
porter_last_name     0
dtype: int64
Duplicate rows:  0


In [128]:
# Clean up and create helper columns
# fix the column name typo, build full names and convert the date column.
df = df.rename(columns={"Exact_amount_paid" : "exact_amount"})
df["farmer"] = df["farmer_first_name"] + " " + df["farmer_last_name"]
df["porter"] = df["porter_first_name"] + " " + df["porter_last_name"]
df["payment_date"] = pd.to_datetime(df["payment_date"])
df

,farmer_first_name,farmer_last_name,amount,payment_date,status,Exact_amout_paid,litres,price_per_litre,porter_first_name,porter_last_name,farmer,porter
0,David,Maina,5730.0,2026-06-05,PAID,2730.0,45.5,60.0,Simon,Ouma,David Maina,Simon Ouma
1,David,Maina,5730.0,2026-06-05,PAID,3000.0,50.0,60.0,Simon,Ouma,David Maina,Simon Ouma
2,Stella,Nyaboke,4800.0,2026-06-05,PAID,2280.0,38.0,60.0,Simon,Ouma,Stella Nyaboke,Simon Ouma
3,Stella,Nyaboke,4800.0,2026-06-05,PAID,2520.0,42.0,60.0,Simon,Ouma,Stella Nyaboke,Simon Ouma
4,Richard,Anyango,6600.0,2026-06-05,PAID,3120.0,52.0,60.0,Ezra,Nehemiah,Richard Anyango,Ezra Nehemiah
5,Richard,Anyango,6600.0,2026-06-05,PAID,3480.0,58.0,60.0,Ezra,Nehemiah,Richard Anyango,Ezra Nehemiah
6,Lilian,Nasimiyu,4830.0,2026-06-05,PAID,2490.0,41.5,60.0,Ezra,Nehemiah,Lilian Nasimiyu,Ezra Nehemiah
7,Lilian,Nasimiyu,4830.0,2026-06-05,PAID,2340.0,39.0,60.0,Ezra,Nehemiah,Lilian Nasimiyu,Ezra Nehemiah
8,Duncan,Kiplagat,7380.0,2026-06-05,PAID,3600.0,60.0,60.0,Job,Mwangi,Duncan Kiplagat,Job Mwangi
9,Duncan,Kiplagat,7380.0,2026-06-05,PAID,3780.0,63.0,60.0,Job,Mwangi,Duncan Kiplagat,Job Mwangi
